# SQL Interview Practice — Day 16
## 15 Questions: JOINs + SELF JOIN + Window Functions + CTE + Business Logic

**Level:** Data Analyst — 2–3 years experience  
**No solutions included. Write the SQL yourself.**

In [1]:
import pandas as pd
import pandasql
from pandasql import sqldf



employees = pd.DataFrame({
    "employee_id":[101,102,103,104,105,106,107,108,109,110,111,112],
    "employee_name":["Amit","Neha","Rahul","Priya","Vikas","Sneha","Arjun","Pooja","Karan","Riya","Manish","Anjali"],
    "department_id":[10,10,20,20,30,30,10,20,30,10,40,40],
    "manager_id":[None,101,101,103,103,105,101,103,105,102,105,111],
    "salary":[95000,78000,72000,88000,65000,82000,110000,76000,99000,85000,70000,91000],
    "join_date":pd.to_datetime(["2021-04-10","2022-07-15","2023-01-20","2021-11-05","2024-02-12","2023-08-18","2020-06-01","2024-01-10","2022-03-22","2025-02-14","2023-05-30","2024-09-01"])
})

departments = pd.DataFrame({
    "department_id":[10,20,30,40,50],
    "department_name":["Sales","IT","Finance","HR","Marketing"],
    "location":["Delhi","Bengaluru","Mumbai","Delhi","Pune"]
})

customers = pd.DataFrame({
    "customer_id":[1,2,3,4,5,6,7],
    "customer_name":["Aarav","Diya","Kabir","Ishita","Rohan","Meera","Vivaan"],
    "city":["Delhi","Mumbai","Pune","Delhi","Bengaluru","Mumbai","Pune"]
})

orders = pd.DataFrame({
    "order_id":[1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014],
    "customer_id":[1,2,1,3,4,2,5,1,6,5,2,3,7,1],
    "order_date":pd.to_datetime(["2025-01-05","2025-01-10","2025-02-12","2025-02-20","2025-03-01","2025-03-18","2025-04-02","2025-04-15","2025-05-07","2025-05-20","2025-06-10","2025-06-15","2025-07-01","2025-07-18"]),
    "amount":[1200,2500,1800,3200,900,4100,1500,2700,3600,2200,2800,3500,1700,4500],
    "status":["Completed","Completed","Cancelled","Completed","Completed","Completed","Completed","Completed","Completed","Cancelled","Completed","Completed","Completed","Completed"]
})

products = pd.DataFrame({
    "product_id":[501,502,503,504,505,506,507],
    "product_name":["Laptop","Phone","Tablet","Monitor","Keyboard","Headphones","Mouse"],
    "category":["Electronics","Electronics","Electronics","Electronics","Accessories","Accessories","Accessories"],
    "price":[60000,30000,25000,18000,2500,4000,1200]
})

order_items = pd.DataFrame({
    "order_id":[1001,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014],
    "product_id":[501,505,502,503,504,505,502,507,506,504,501,502,503,507,501],
    "quantity":[1,2,1,1,2,1,2,3,1,1,1,2,1,2,1]
})

monthly_sales = pd.DataFrame({
    "month":pd.to_datetime(["2025-01-01","2025-02-01","2025-03-01","2025-04-01","2025-05-01","2025-06-01","2025-07-01"]),
    "revenue":[120000,145000,132000,160000,158000,175000,190000]
})

## Q1 — JOIN + Aggregation
Find the total **completed order amount for each customer**. Include customers with no completed orders.
Return customer_id, customer_name, total_completed_amount.

In [2]:
sqldf("""
select c.customer_id as customer_id , c.customer_name as name , sum(o.amount) as toatl_complted_order
from customers c
left join orders o on c.customer_id = o.customer_id 
where status == 'Completed'
group by c.customer_id

""")

,customer_id,name,toatl_complted_order
0,1,Aarav,8400
1,2,Diya,9400
2,3,Kabir,6700
3,4,Ishita,900
4,5,Rohan,1500
5,6,Meera,3600
6,7,Vivaan,1700


## Q2 — JOIN + Business Logic
Find customers whose **total completed spending is greater than ₹5,000**. Return customer_id, customer_name, total_spending.

In [3]:
sqldf("""
select c.customer_id as customer_id , c.customer_name as customer_name , sum(o.amount) as total_spending
from customers c
left join orders o on c.customer_id = o.customer_id
group by c.customer_id
having sum(o.amount) > 5000


""")

,customer_id,customer_name,total_spending
0,1,Aarav,10200
1,2,Diya,9400
2,3,Kabir,6700


## Q3 — SELF JOIN
Find every employee and their manager's name. Employees without a manager should still appear.

In [4]:
sqldf("""
select e.employee_id ,
        e.manager_id ,
        e.employee_name as emp_name ,
        m.employee_name as manager_name
from employees e
left join  employees m on e.manager_id = m.employee_id


""")

,employee_id,manager_id,emp_name,manager_name
0,101,NaN,Amit,NaN
1,102,101.0,Neha,Amit
2,103,101.0,Rahul,Amit
3,104,103.0,Priya,Rahul
4,105,103.0,Vikas,Rahul
5,106,105.0,Sneha,Vikas
6,107,101.0,Arjun,Amit
7,108,103.0,Pooja,Rahul
8,109,105.0,Karan,Vikas
9,110,102.0,Riya,Neha


## Q4 — SELF JOIN + Business Logic
Find employees whose salary is **greater than their manager's salary**. Return employee, employee salary, manager, manager salary.

In [5]:
sqldf("""
select 
    e.employee_name as emp_name , 
    e.salary as emp_salary ,
    m.employee_name as manager_name,
    m.salary as manager_salary
from employees e
join employees m on e.manager_id = m.employee_id 
where e.salary > m.salary


""")

,emp_name,emp_salary,manager_name,manager_salary
0,Priya,88000,Rahul,72000
1,Sneha,82000,Vikas,65000
2,Arjun,110000,Amit,95000
3,Pooja,76000,Rahul,72000
4,Karan,99000,Vikas,65000
5,Riya,85000,Neha,78000
6,Manish,70000,Vikas,65000
7,Anjali,91000,Manish,70000


## Q5 — Window Function
Rank employees by salary **within each department**. Return employee_name, department_id, salary, salary_rank. Handle ties appropriately.

In [6]:
sqldf("""
SELECT
    employee_name,
    department_id,
    salary,
    DENSE_RANK() OVER (
        PARTITION BY department_id
        ORDER BY salary DESC
    ) AS salary_rank
FROM employees
""")

,employee_name,department_id,salary,salary_rank
0,Arjun,10,110000,1
1,Amit,10,95000,2
2,Riya,10,85000,3
3,Neha,10,78000,4
4,Priya,20,88000,1
5,Pooja,20,76000,2
6,Rahul,20,72000,3
7,Karan,30,99000,1
8,Sneha,30,82000,2
9,Vikas,30,65000,3


## Q6 — Window + Business Logic
Find the **highest-paid employee(s) in each department**. Return all ties.

In [7]:
sqldf("""
select * 
from (select *,
        dense_rank()
        over(partition by department_id order by salary desc) as rn
        from employees) t
where rn = 1


""")

,employee_id,employee_name,department_id,manager_id,salary,join_date,rn
0,107,Arjun,10,101.0,110000,2020-06-01 00:00:00.000000,1
1,104,Priya,20,103.0,88000,2021-11-05 00:00:00.000000,1
2,109,Karan,30,105.0,99000,2022-03-22 00:00:00.000000,1
3,112,Anjali,40,111.0,91000,2024-09-01 00:00:00.000000,1


## Q7 — CTE + JOIN
Create a CTE for each customer's total completed spending. Join it to customers and return customers whose spending is **above the average customer spending**.

In [8]:
sqldf("""
with my_cte as (
select c.customer_id,
        c.customer_name,
        sum(o.amount) as total_spending
from customers c
left join orders o on c.customer_id = o.customer_id 
group by c.customer_id 

)
select * from my_cte
where total_spending > (select avg(total_spending) 
                            from my_cte)

""")

,customer_id,customer_name,total_spending
0,1,Aarav,10200
1,2,Diya,9400
2,3,Kabir,6700


## Q8 — CTE + Window
Using orders, calculate monthly completed revenue. Then use a window function to return month, monthly_revenue, previous_month_revenue, and month_over_month_change.

In [9]:
employees

,employee_id,employee_name,department_id,manager_id,salary,join_date
0,101,Amit,10,NaN,95000,2021-04-10
1,102,Neha,10,101.0,78000,2022-07-15
2,103,Rahul,20,101.0,72000,2023-01-20
3,104,Priya,20,103.0,88000,2021-11-05
4,105,Vikas,30,103.0,65000,2024-02-12
5,106,Sneha,30,105.0,82000,2023-08-18
6,107,Arjun,10,101.0,110000,2020-06-01
7,108,Pooja,20,103.0,76000,2024-01-10
8,109,Karan,30,105.0,99000,2022-03-22
9,110,Riya,10,102.0,85000,2025-02-14


## Q9 — Window + Business Logic
For every employee calculate department average salary and classify them as Above Average, Equal Average, or Below Average.

In [10]:
# Write your SQL query here
sqldf("""
with my_cte as (
select e.employee_id ,e.employee_name, d.department_name , avg(e.salary) as dept_avg_salary
from employees e 
left join departments d on e.department_id = d.department_id
group by e.employee_id

)
select * from my_cte

""")

,employee_id,employee_name,department_name,dept_avg_salary
0,101,Amit,Sales,95000.0
1,102,Neha,Sales,78000.0
2,103,Rahul,IT,72000.0
3,104,Priya,IT,88000.0
4,105,Vikas,Finance,65000.0
5,106,Sneha,Finance,82000.0
6,107,Arjun,Sales,110000.0
7,108,Pooja,IT,76000.0
8,109,Karan,Finance,99000.0
9,110,Riya,Sales,85000.0


## Q10 — JOIN + Window
Find the **top 2 customers by completed spending in each city**. Return city, customer, total_spending, and rank_in_city. Handle ties.

In [11]:
sqldf("""SELECT
    customer_name,
    city,
    total_spending
FROM (
    SELECT
        c.customer_name,
        c.city,
        SUM(o.amount) AS total_spending,
        DENSE_RANK() OVER (
            PARTITION BY c.city
            ORDER BY SUM(o.amount) DESC
        ) AS rn
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name,
        c.city
) t
WHERE rn <= 2

""")

,customer_name,city,total_spending
0,Rohan,Bengaluru,3700
1,Aarav,Delhi,10200
2,Ishita,Delhi,900
3,Diya,Mumbai,9400
4,Meera,Mumbai,3600
5,Kabir,Pune,6700
6,Vivaan,Pune,1700


## Q11 — SELF JOIN + Business Logic
Find pairs of employees in the **same department**. Do not pair an employee with themselves and do not return the same pair twice.

In [12]:
sqldf("""
WITH my_cte AS (
    SELECT
        e.employee_id,
        e.employee_name,
        e.department_id,
        d.department_name
    FROM employees e
    LEFT JOIN departments d
        ON e.department_id = d.department_id
)

SELECT
    a.employee_name AS employee_1,
    b.employee_name AS employee_2,
    a.department_name
FROM my_cte a
JOIN my_cte b
    ON a.department_id = b.department_id
   AND a.employee_id < b.employee_id
""")

,employee_1,employee_2,department_name
0,Amit,Neha,Sales
1,Amit,Arjun,Sales
2,Amit,Riya,Sales
3,Neha,Arjun,Sales
4,Neha,Riya,Sales
5,Rahul,Priya,IT
6,Rahul,Pooja,IT
7,Priya,Pooja,IT
8,Vikas,Sneha,Finance
9,Vikas,Karan,Finance


In [13]:
sqldf("""
with my_cte as (
select e.* ,d.*
from employees e
left join departments d
on e. department_id = d.department_id
)
select a.employee_name , b.department_name
from my_cte a
join my_cte b
where a.department_id = b.department_id
and a.employee_id < b.employee_id

""")

,employee_name,department_name
0,Amit,Sales
1,Amit,Sales
2,Amit,Sales
3,Neha,Sales
4,Neha,Sales
5,Rahul,IT
6,Rahul,IT
7,Priya,IT
8,Vikas,Finance
9,Vikas,Finance


## Q12 — CTE + JOIN + Business Logic
Find departments whose **average employee salary is greater than the company's overall average salary**. Return department_id, department_name, department_avg_salary.

In [14]:
sqldf("""
with my_cte as (
    select  e.department_id , d.department_name, avg(salary) as avg_salary
    from employees e
    left join departments d on e.department_id  = d.department_id
    group by e.department_id
)
select * from my_cte
where avg_salary > (select avg(Salary) 
                    from employees)
""")

,department_id,department_name,avg_salary
0,10,Sales,92000.0


## Q13 — CTE + Window + Business Logic
For each customer calculate completed order count and total spending, rank customers by spending, then classify: rank 1–2 = High Value, rank 3–5 = Medium Value, rank >5 = Low Value.

In [15]:
sqldf("""
with my_cte as (select customer_name , total_orders , total_spending , rn
from (
select c.customer_name, count(order_id) as total_orders, sum(amount) as total_spending,
dense_rank()
over(order by sum(amount) desc) as rn 
from customers c
left join orders o on c.customer_id = o.customer_id
where status = 'Completed'
group by c.customer_id ,c.customer_name)

)
select customer_name , total_orders , total_spending,
case 
    when rn >= 1 and rn <=2 then  'High_value'
    when rn >= 3 and rn <=5 then 'medium_value'
    else 'Low Value'
end as buket 
from my_cte
""")

,customer_name,total_orders,total_spending,buket
0,Diya,3,9400,High_value
1,Aarav,3,8400,High_value
2,Kabir,2,6700,medium_value
3,Meera,1,3600,medium_value
4,Vivaan,1,1700,medium_value
5,Rohan,1,1500,Low Value
6,Ishita,1,900,Low Value


In [16]:
employees

,employee_id,employee_name,department_id,manager_id,salary,join_date
0,101,Amit,10,NaN,95000,2021-04-10
1,102,Neha,10,101.0,78000,2022-07-15
2,103,Rahul,20,101.0,72000,2023-01-20
3,104,Priya,20,103.0,88000,2021-11-05
4,105,Vikas,30,103.0,65000,2024-02-12
5,106,Sneha,30,105.0,82000,2023-08-18
6,107,Arjun,10,101.0,110000,2020-06-01
7,108,Pooja,20,103.0,76000,2024-01-10
8,109,Karan,30,105.0,99000,2022-03-22
9,110,Riya,10,102.0,85000,2025-02-14


## Q14 — Advanced Mixed Business Case
Find employees satisfying BOTH: (1) salary is above their department average, and (2) salary is lower than their manager's salary. Return employee salary, department average, manager name and manager salary. Combine CTE + SELF JOIN + aggregation/window logic.

In [17]:
sqldf("""
WITH dept_avg AS (
    SELECT
        department_id,
        AVG(salary) AS dept_avg_salary
    FROM employees
    GROUP BY department_id
)

SELECT
    e.employee_id,
    e.employee_name,
    e.salary AS employee_salary,
    d.dept_avg_salary,
    m.employee_name AS manager_name,
    m.salary AS manager_salary
FROM employees e
JOIN dept_avg d
    ON e.department_id = d.department_id
JOIN employees m
    ON e.manager_id = m.employee_id
WHERE e.salary > d.dept_avg_salary
  AND e.salary < m.salary
ORDER BY e.department_id, e.salary DESC
""")


,employee_id,employee_name,employee_salary,dept_avg_salary,manager_name,manager_salary


## Q15 — Advanced Interview Challenge
Find the **top 2 products by revenue within each product category**. Join products, order_items and orders; use only Completed orders; revenue = quantity × product price; calculate product revenue; rank within category; return ranks 1 and 2; handle ties. Combine JOIN + CTE + Window Function + Business Logic.

In [18]:
sqldf("""
WITH revenue_cte AS (
    SELECT
        p.category,
        p.product_id,
        p.product_name,
        SUM(oi.quantity * p.price) AS revenue,
        DENSE_RANK() OVER (
            PARTITION BY p.category
            ORDER BY SUM(oi.quantity * p.price) DESC
        ) AS rnk
    FROM products p
    JOIN order_items oi
        ON p.product_id = oi.product_id
    JOIN orders o
        ON oi.order_id = o.order_id
    WHERE o.status = 'Completed'
    GROUP BY
        p.category,
        p.product_id,
        p.product_name
)
SELECT *
FROM revenue_cte
WHERE rnk <= 2;
""")

,category,product_id,product_name,revenue,rnk
0,Accessories,505,Keyboard,7500,1
1,Accessories,507,Mouse,6000,2
2,Electronics,502,Phone,150000,1
3,Electronics,501,Laptop,120000,2
